
In this activity, we will analyze a dataset labeled by multiple annotators (e.g. collected via crowdsourcing). Consider the following scenario. We have a small classification dataset (3 classes, 2-dimensional features). Each sample was independently labeled by one of 50 annotators; some annotatators were not able to label a specific sample, resulting in an N/A label. There is some probability of error in each annotated label, where the probability is determined by the underlying quality of the annotator. You have two goals:
1. Find 10 annotators with the largest number of errors.
2. Train a classification model.

In [ ]:
!pip install cleanlab
# This automatically installs other required packages like numpy, pandas, sklearn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.8/349.8 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 69.7 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have nump

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
X = pd.read_csv('./csc790-f25-activity8-features.csv', index_col=False)
multiannotator_labels = pd.read_csv('./csc790-f25-activity8-labels.csv', index_col=False)

In [ ]:
# df = pd.DataFrame(true_labels)
# df.to_csv('./csc790-f25-activity8-true_labels.csv', index=False)

### Baseline method
1. Implement code to randomly select a label from annotations (pay attention, there are NAs when annotator did not know what to assign).
2. Estimate the quality of annotator (how accurate their labels tend to be overall). Who are the worst 10 annotators?

In [ ]:

# randomly select a label for each instance (ignoring NaN values)

def random_select_labels(labels_df):
    #create an array to store the randomly selected labels
    selected_labels = []

    for i in range(len(labels_df)):
        # Get the row of annotations for a specific instance
        labels = labels_df.iloc[i].dropna().values  # Drop NaNs
        if len(labels) > 0:
            selected_label = np.random.choice(labels)  # Randomly choose a label
        else:
            selected_label = np.random.choice([0, 1, 2]) # If no labels available,randomly select from [0, 1, 2]
        selected_labels.append(selected_label)

    return selected_labels


In [ ]:
# Apply the function to the annotations data
random_labels = random_select_labels(multiannotator_labels)

# Add the selected labels to a new dataframe
random_labels_df = pd.DataFrame(random_labels, columns=['randomly_selected_label'])
random_labels_df.head()

#df_random_labels.to_csv('./csc790-f25-activity8-random_labels.csv', index=False)


,randomly_selected_label
0,2.0
1,2.0
2,2.0
3,2.0
4,1.0


In [ ]:
# Assuming `true_labels` is available, which should be the ground truth
#true_labels = pd.read_csv('./csc790-f25-activity8-true_labels.csv')

# estimate the quality of annotators
def estimate_quality(annotator_labels_df, true_labels):
    annotator_accuracies = {}

    for annotator in annotator_labels_df.columns:
        # Get the labels for a specific annotator
        annotator_values = annotator_labels_df[annotator].values

        # Calculate accuracy by comparing with true labels
        correct = (annotator_values == true_labels.values.flatten())  # Compare element-wise
        accuracy = np.mean(correct)  # Calculate accuracy as the proportion of correct matches
        annotator_accuracies[annotator] = accuracy

    # Sort the annotators by their accuracy in ascending order and get the worst 10
    worst_annotators = sorted(annotator_accuracies.items(), key=lambda x: x[1])[:10]

    return worst_annotators


In [ ]:
# Get the worst 10 annotators based on accuracy
worst_annotators = estimate_quality(multiannotator_labels, random_labels_df)

# Print the worst 10 annotators
print("Worst 10 annotators based on accuracy:")
for annotator, accuracy in worst_annotators:
    print(f"{annotator}: {accuracy:.4f}")


Worst 10 annotators based on accuracy:
A0038: 0.0368
A0044: 0.0368
A0045: 0.0368
A0002: 0.0468
A0012: 0.0468
A0025: 0.0468
A0028: 0.0468
A0041: 0.0502
A0046: 0.0502
A0047: 0.0502


## Your heuristic for finding consensus labels

1. Implement a method to create the most likely label for the dataset. Save these labels.
2. Estimate the quality of annotator (how accurate their labels tend to be overall). Who are the worst 10 annotators?

In [ ]:
from scipy.stats import mode

# most likely labels from consensus - select a label for each instance (ignoring NaN values)

def most_likely_labels(labels_df):
    #create an array to store the randomly selected labels
    selected_labels = []

    for i in range(len(labels_df)):
        # Get the row of annotations for a specific instance
        labels = labels_df.iloc[i].dropna().values  # Drop NaNs
        if len(labels) > 0:
            # selected_label = mode(labels).mode[0]
            selected_label = mode(labels, keepdims=True)[0][0] #max(labels) # select the max - consensus
        else:
            selected_label = np.random.choice([0, 1, 2]) # If no labels available,randomly select from [0, 1, 2]
        selected_labels.append(selected_label)

    return selected_labels


In [ ]:
# Apply the function to the annotations data
consensus_labels = most_likely_labels(multiannotator_labels)

# Add the selected labels to a new dataframe
consensus_labels_df = pd.DataFrame(consensus_labels, columns=['randomly_selected_label'])
consensus_labels_df.head()


,randomly_selected_label
0,2.0
1,2.0
2,2.0
3,1.0
4,1.0


In [ ]:
# Get the worst 10 annotators based on accuracy
worst_annotators_consensus = estimate_quality(multiannotator_labels, consensus_labels_df)

# Print the worst 10 annotators
print("Worst 10 annotators based on accuracy:")
for annotator, accuracy in worst_annotators_consensus:
    print(f"{annotator}: {accuracy:.4f}")

Worst 10 annotators based on accuracy:
A0044: 0.0301
A0038: 0.0401
A0045: 0.0435
A0048: 0.0435
A0046: 0.0502
A0041: 0.0535
A0047: 0.0535
A0013: 0.0569
A0040: 0.0569
A0002: 0.0602


## Cleanlab multiannotator library

Use cleanlab library: https://docs.cleanlab.ai/stable/tutorials/multiannotator.html
See the source code for implementation: https://github.com/cleanlab/cleanlab/blob/master/cleanlab/multiannotator.py

1. Implement a method to create the most likely label for the dataset. Save these labels.
2. Estimate the quality of annotator (how accurate their labels tend to be overall). Who are the worst 10 annotators?

In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from cleanlab.multiannotator import get_label_quality_multiannotator, get_majority_vote_label

# Convert to numpy array (with NaNs for missing)
# Cleanlab expects shape (num_examples, num_annotators) with np.nan for missing labels.
label_array = multiannotator_labels.to_numpy()

# 1. Baseline consensus via majority vote
majority_labels = get_majority_vote_label(label_array)

# 2. Train classifier & get predicted probabilities (via cross-val)
clf = LogisticRegression(max_iter=1000, multi_class='multinomial')
# Use majority_labels as pseudo-ground truth
pred_probs = cross_val_predict(clf, X, majority_labels, cv=5, method='predict_proba')

# 3. Use Cleanlab multiannotator to get refined consensus and annotator stats
results = get_label_quality_multiannotator(label_array, pred_probs)

# results is a dict containing:
#  - “label_quality” DataFrame: includes consensus_label, consensus_quality_score, etc.
#  - “annotator_stats” DataFrame: annotator_quality, agreement_with_consensus, etc.
#  - “detailed_label_quality” DataFrame: quality per (example, annotator) label
label_quality = results['label_quality']
annotator_stats = results['annotator_stats']
detailed = results['detailed_label_quality']

# 4. Extract consensus labels and annotator qualities
consensus_labels = label_quality['consensus_label'].to_numpy()
annotator_qualities = annotator_stats['annotator_quality']

# 5. Get worst 10 annotators
worst_10 = annotator_stats.nsmallest(10, 'annotator_quality')
print("Worst 10 annotators (by Cleanlab):")
print(worst_10)


Worst 10 annotators (by Cleanlab):
    annotator_quality  agreement_with_consensus  worst_class  \
42           0.357891                  0.363636            2   
41           0.368329                  0.342857            2   
47           0.370843                  0.375000            1   
43           0.389990                  0.428571            2   
40           0.400404                  0.464286            2   
45           0.410808                  0.448276            1   
39           0.435001                  0.481481            2   
44           0.438966                  0.461538            2   
49           0.443339                  0.500000            2   
48           0.471697                  0.527778            2   

    num_examples_labeled  
42                    33  
41                    35  
47                    32  
43                    21  
40                    28  
45                    29  
39                    27  
44                    26  
49               

## Evaluation
1. Train a classification model using consensus labels from your heuristic. Compute 5-fold cross-validated accuracy.
2. Train a second model using cleanlab's labels. Compute 5-fold cross-validated accuracy.
3. Train a baseline model with randomly selected labels. Compute 5-fold cross-validated accuracy.
4. Load true labels: csc790-f25-activity8-true_labels.csv. For each method, compute the number of correctly computed consensus labels.

5. Run small experiments to evaluate different methods for consensus labeling, in addition to simple counting/averaging of correct consensus labels.

In [ ]:

from sklearn.model_selection import cross_val_score

# Classifier
clf = LogisticRegression(max_iter=1000, multi_class='multinomial')

# train/test using random labels
y_random = random_labels_df['randomly_selected_label'].astype(int)
acc_random = cross_val_score(clf, X, y_random, cv=5, scoring='accuracy')
print("Random labels - Mean CV accuracy:", np.mean(acc_random))

# train/test using heuristic consensus labels
y_consensus = consensus_labels_df['randomly_selected_label'].astype(int)
acc_consensus = cross_val_score(clf, X, y_consensus, cv=5, scoring='accuracy')
print("Heuristic consensus labels - Mean CV accuracy:", np.mean(acc_consensus))

# train/test using Cleanlab consensus labels
y_cleanlab = label_quality['consensus_label'].astype(int)
acc_cleanlab = cross_val_score(clf, X, y_cleanlab, cv=5, scoring='accuracy')
print("Cleanlab consensus labels - Mean CV accuracy:", np.mean(acc_cleanlab))


Random labels - Mean CV accuracy: 0.6357627118644067
Heuristic consensus labels - Mean CV accuracy: 0.7996045197740111
Cleanlab consensus labels - Mean CV accuracy: 0.9899435028248588


In [ ]:
# 1️⃣ Load true labels
true_labels = pd.read_csv('./csc790-f25-activity8-true_labels.csv')

# Ensure same length and correct type
true_labels = true_labels.squeeze().astype(int)  # make sure it’s a Series of ints

# 2️⃣ Compute accuracy for each labeling approach

# Random
random_correct = np.sum(random_labels_df['randomly_selected_label'].astype(int).values == true_labels.values)
random_acc = random_correct / len(true_labels)

# Heuristic consensus
consensus_correct = np.sum(consensus_labels_df['randomly_selected_label'].astype(int).values == true_labels.values)
consensus_acc = consensus_correct / len(true_labels)

# Cleanlab
cleanlab_correct = np.sum(label_quality['consensus_label'].astype(int).values == true_labels.values)
cleanlab_acc = cleanlab_correct / len(true_labels)

# 3️⃣ Print results
print("Number of correctly predicted labels:")
print(f"Random: {random_correct} / {len(true_labels)} ({random_acc:.2%})")
print(f"Heuristic consensus: {consensus_correct} / {len(true_labels)} ({consensus_acc:.2%})")
print(f"Cleanlab consensus: {cleanlab_correct} / {len(true_labels)} ({cleanlab_acc:.2%})")


Number of correctly predicted labels:
Random: 199 / 299 (66.56%)
Heuristic consensus: 259 / 299 (86.62%)
Cleanlab consensus: 290 / 299 (96.99%)


##### 5. Run small experiments to evaluate different methods for consensus labeling, in addition to simple counting/averaging of correct consensus labels.

In [ ]:
# A
### Implementing alternative consensus rules - Weighted vote

# Suppose you already have annotator qualities
# Example: annotator_qualities from Cleanlab or heuristic
# For simplicity, random example if not available:
#annotator_qualities = np.ones(label_array.shape[1])

weighted_labels = []
for i in range(label_array.shape[0]):
    labels = label_array[i]
    mask = ~np.isnan(labels)
    if np.sum(mask) == 0:
        weighted_labels.append(np.random.choice([0,1,2]))
        continue
    # Apply weights
    weighted_counts = {}
    for idx, label in enumerate(labels):
        if mask[idx]:
            weighted_counts[label] = weighted_counts.get(label,0) + annotator_qualities[idx]
    # Pick label with max weighted count
    consensus = max(weighted_counts, key=weighted_counts.get)
    weighted_labels.append(consensus)

weighted_labels = np.array(weighted_labels)
#print(weighted_labels)


In [ ]:
# B
# Averaging (for numeric labels)
avg_labels = []
for i in range(label_array.shape[0]):
    labels = label_array[i]
    mask = ~np.isnan(labels)
    if np.sum(mask) == 0:
        avg_labels.append(np.random.choice([0,1,2]))
    else:
        avg_labels.append(int(np.round(np.mean(labels[mask]))))

avg_labels = np.array(avg_labels)
#print(avg_labels)


In [ ]:
'''
Evaluate each method against true labels
'''

from sklearn.metrics import accuracy_score
#true_labels = pd.read_csv('./csc790-f25-activity8-true_labels.csv').squeeze().astype(int)

methods = {
    "Random": random_labels_df['randomly_selected_label'].astype(int).values,
    "Mode Consensus": consensus_labels_df['randomly_selected_label'].astype(int).values,
    "Cleanlab": label_quality['consensus_label'].astype(int).values,
    "Weighted vote": weighted_labels,
    "Average vote": avg_labels
}

for name, y_pred in methods.items():
    acc = accuracy_score(true_labels, y_pred)
    print(f"{name} consensus accuracy: {acc:.2%}")


Random consensus accuracy: 66.56%
Mode Consensus consensus accuracy: 86.62%
Cleanlab consensus accuracy: 96.99%
Weighted vote consensus accuracy: 87.29%
Average vote consensus accuracy: 60.54%
